In [ ]:
!pip install pymupdf

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "Enter your gemini api key here"

import torch
print(f"GPU available: {torch.cuda.is_available()}")

import gradio as gr
import tempfile, shutil, traceback, os
import fitz
from llama_index.core import VectorStoreIndex, Settings, Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash-lite")
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5", device="cuda")
Settings.transformations = [SentenceSplitter(chunk_size=1024, chunk_overlap=100)]

engine = None

def extract_pdf_text(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

def load_files(files):
    global engine
    if not files:
        return "⚠️ Please upload at least one file!"
    try:
        documents = []
        report = []
        for f in files:
            name = os.path.basename(f.name)
            text = extract_pdf_text(f.name)
            report.append(f"{name}: {len(text)} chars")
            if len(text.strip()) > 100:
                documents.append(Document(text=text, metadata={"filename": name}))
        if not documents:
            return "❌ No readable text extracted! PDFs may be scanned images (need OCR).\n" + "\n".join(report)
        index = VectorStoreIndex.from_documents(documents, show_progress=True)
        engine = index.as_query_engine(similarity_top_k=10, response_mode="compact")
        return "✅ Loaded!\n" + "\n".join(report)
    except Exception as e:
        return f"❌ Error: {traceback.format_exc()}"

def ask(question):
    if engine is None:
        return "⚠️ Load files first!"
    if not question.strip():
        return "⚠️ Type a question!"
    try:
        return str(engine.query(question))
    except Exception as e:
        return f"❌ Error: {traceback.format_exc()}"

with gr.Blocks() as app:
    gr.Markdown("# 📂 Ask Your Documents")
    with gr.Row():
        files = gr.File(file_count="multiple", label="📁 Upload Files")
        load_btn = gr.Button("Load Files", variant="primary")
    status = gr.Textbox(label="Status", interactive=False, lines=8)
    question = gr.Textbox(label="❓ Ask a question")
    ask_btn = gr.Button("Ask", variant="primary")
    answer = gr.Textbox(label="💬 Answer", interactive=False, lines=6)
    load_btn.click(load_files, inputs=files, outputs=status)
    ask_btn.click(ask, inputs=question, outputs=answer)

app.launch(share=True)